# Phase 3 — HFT Simulation (3-day isolated scenarios)

Runs each of the four HFT scenarios (`normal`, `vol_spike`, `hft_onesided`, `hft_offline`) for **3 days** at `dt = 0.05 s`.

HFT quotes at **2 000 EUR depth** — with mean client order ≈ 2 700 EUR (Pareto α=1.5),  
~35% of client orders spill past the HFT onto our ladder.  

Each scenario prints its own full report.  A comparison table is shown at the end.

In [1]:
import sys, copy, time, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append('../src')

import utils.stock_simulation as stock_mod
import utils.market_simulator as market_mod
import utils.order_book.order_book_impl as book_mod
import utils.market_maker.quoter as quoter_mod
import utils.report.pnl_tracker as pnl_mod
import utils.report.controller as ctrl_mod
import utils.client_flow.flow_generator as flow_mod
import utils.hft.hft_config as hft_cfg_mod
import utils.hft.hft_agent as hft_agent_mod
import utils.hft.scenarios as hft_sc_mod

for mod in [stock_mod, market_mod, book_mod, quoter_mod, pnl_mod,
            ctrl_mod, flow_mod, hft_cfg_mod, hft_agent_mod, hft_sc_mod]:
    importlib.reload(mod)

from utils.stock_simulation import Stock
from utils.market_simulator.market import Market
from utils.order_book.order_book_impl import Order_book
from utils.market_maker.quoter import Quoter, QuoterConfig
from utils.report.pnl_tracker import PnLTracker
from utils.report.controller import Controller
from utils.client_flow.flow_generator import ClientFlowGenerator
from utils.hft.hft_config import HFTConfig
from utils.hft.scenarios import ISOLATED, HFTState

print('Modules loaded.')

Modules loaded.


In [ ]:
# ── Simulation constants ──────────────────────────────────────────────────────
SEED    = 42
CAPITAL = 1_000_000.0
N_DAYS  = 1
DT_SEC  = 0.05          # 0.05 s → 1 step = HFT latency (50 ms)
WINDOW  = 7_200         # vol window for market spread builder

# ── HFT quoting size ─────────────────────────────────────────────────────────
# Mean client order ≈ 2700 EUR (Pareto α=1.5, min=1k, max=100k).
# At 2000 EUR depth, ~35% of client orders exceed HFT and spill to our ladder.
HFT_DEPTH_EUR = 2_000

n_steps = round(N_DAYS * 86_400 / DT_SEC)
print(f'{N_DAYS} day at dt={DT_SEC}s → {n_steps:,} steps')
print(f'Estimated runtime: ~{n_steps / 12_000 / 60:.0f} min per scenario')

In [3]:
def build_simulation(seed=SEED):
    """Build fresh markets, book, quoter, and client flow (same seed = comparable runs)."""
    np.random.seed(seed)

    # Stock
    stock = Stock(drift=0.0, vol=0.20)
    stock.simulate_garch(n_days=N_DAYS, dt_seconds=DT_SEC,
                         alpha=0.05, beta=0.94, lam=756, sigma_J=0.005)

    # Reference markets B and C
    mkt_B = Market(stock=stock)
    mkt_B.generate_noised_mid_price()
    mkt_B.build_spread(option='Skew', window_size=WINDOW, alpha=0.5,
                       gamma=0.3, ema_span=5_000, threshold=3)
    mkt_B.generate_depth(mean_eur=500_000)

    mkt_C = copy.deepcopy(mkt_B)
    mkt_C.build_spread(option='Adaptive', window_size=WINDOW)
    mkt_C.generate_depth(mean_eur=200_000)

    # Quoter config — identical to Phase 2 (no adaptation for HFT presence)
    qcfg = QuoterConfig(
        requote_threshold_spread_fraction=0.25,
        delta_limit=0.90,
        hedge_partial_limit=0.80,
        emergency_penalty_multiplier=5.0,
    )

    book = Order_book()
    mm   = Quoter(mkt_B, mkt_C, config=qcfg, capital_K=CAPITAL)
    book.register_quoter_listener(mm.on_fill)

    gen = ClientFlowGenerator(seed=seed)
    client_fn = lambda step, t, mid, bid, ask, dt: \
        gen.generate_step(mid_price=mid, best_bid=bid, best_ask=ask, dt=dt)

    return mkt_B, mkt_C, book, mm, client_fn

## Isolated scenario runs

| Scenario | What it tests |
|---|---|
| `normal` | HFT active all day, standard params — baseline with competition |
| `vol_spike` | HFT more selective: offline above 22% vol, needs 1 bps net spread |
| `hft_onesided` | Forced ONE_SIDED_BID day 0.5–1.5, ONE_SIDED_ASK day 2.0–2.5 |
| `hft_offline` | HFT never quotes (spread_fraction=999) — pure Phase 2 MM baseline |

In [ ]:
import dataclasses

def run_scenario(name):
    """Run one isolated HFT scenario and return (ctrl, log, trades)."""
    iso_cfg, iso_sched = ISOLATED[name]
    mkt_B, mkt_C, book, mm, client_fn = build_simulation()
    hft_cfg = dataclasses.replace(iso_cfg, max_depth_eur=HFT_DEPTH_EUR)
    ctrl = Controller(mkt_B, mkt_C, book, mm, client_fn,
                      hft=True, hft_config=hft_cfg, hft_schedule=iso_sched)
    t0 = time.time()
    ctrl.simulate(limit=n_steps)
    elapsed = time.time() - t0
    log    = ctrl.step_log
    trades = ctrl.trade_history
    mm_fills  = len(trades[~trades['is_hedge']]) if not trades.empty else 0
    hft_fills = int(log['hft_fills_this_step'].sum())
    total     = mm_fills + hft_fills
    print(f'Done in {elapsed:.0f}s  |  MM fills={mm_fills:,}  HFT fills={hft_fills:,}  '
          f'MM share={mm_fills/total:.1%}' if total > 0 else f'Done in {elapsed:.0f}s  |  no fills')
    state_names = {0:'ACTIVE', 1:'ONE_SIDED_BID', 2:'ONE_SIDED_ASK', 3:'OFFLINE'}
    for sv, sn in state_names.items():
        pct = (log['hft_state'] == sv).mean() * 100
        if pct > 0.1:
            print(f'  HFT {sn:<18}: {pct:.1f}% of steps')
    return ctrl, log.copy(), trades.copy()

results = {}

## Scenario 1 — Normal\nHFT fully active, standard profitability threshold, standard vol threshold.

In [ ]:
ctrl_normal, log_normal, trades_normal = run_scenario('normal')
results['normal'] = (log_normal, trades_normal)
ctrl_normal.report()

## Scenario 2 — Vol spike\nHFT more sensitive: offline above 22% annualized vol AND requires higher net spread (1 bps after fees).

In [ ]:
ctrl_vol, log_vol, trades_vol = run_scenario('vol_spike')
results['vol_spike'] = (log_vol, trades_vol)
ctrl_vol.report()

## Scenario 3 — One-sided
HFT is forced ONE_SIDED_BID from hour 4.8 to 10.8 (day 0.2–0.45), then ONE_SIDED_ASK from hour 14.4 to 20.4 (day 0.6–0.85).
Outside those windows the per-side competitiveness check applies (typically ACTIVE).

In [ ]:
ctrl_one, log_one, trades_one = run_scenario('hft_onesided')
results['hft_onesided'] = (log_one, trades_one)
ctrl_one.report()

## Scenario 4 — HFT offline\nHFT never quotes (spread_fraction=999) — baseline: pure Phase 2 MM with no HFT competition.

In [ ]:
ctrl_off, log_off, trades_off = run_scenario('hft_offline')
results['hft_offline'] = (log_off, trades_off)
ctrl_off.report()

## Comparison table — all scenarios

In [ ]:
rows = []
for name, (log, trades) in results.items():
    if trades.empty:
        rows.append({'scenario': name})
        continue

    mm_tr    = trades[~trades['is_hedge']]
    current_mid = float(log['fair_mid'].iloc[-1])
    rep      = PnLTracker.report(trades, current_mid)

    mm_fills  = len(mm_tr)
    hft_fills = int(log['hft_fills_this_step'].sum())
    total     = mm_fills + hft_fills

    rows.append({
        'scenario':       name,
        'total_mtm_pnl':  round(rep['total_mtm_pnl'], 2),
        'inception_pnl':  round(rep['inception_spread_pnl'], 2),
        'total_fees':     round(rep['total_fees'], 2),
        'mm_fills':       mm_fills,
        'hft_fills':      hft_fills,
        'mm_fill_share':  f"{mm_fills/total:.1%}" if total > 0 else 'n/a',
        'avg_mm_size':    round(mm_tr['size'].mean()) if not mm_tr.empty else 0,
        'hft_active_pct': f"{(log['hft_state'] == 0).mean():.1%}",
        'hft_offline_pct':f"{(log['hft_state'] == 3).mean():.1%}",
    })

df_cmp = pd.DataFrame(rows).set_index('scenario')

print('\nPhase 3 — Scenario Comparison')
print('=' * 90)
print(df_cmp.to_string())
print('=' * 90)

## Fill-share plot — MM vs HFT by scenario

In [ ]:
names = list(results.keys())
mm_counts  = []
hft_counts = []

for name in names:
    log, trades = results[name]
    mm_fills  = len(trades[~trades['is_hedge']]) if not trades.empty else 0
    hft_fills = int(log['hft_fills_this_step'].sum())
    mm_counts.append(mm_fills)
    hft_counts.append(hft_fills)

x = np.arange(len(names))
totals = [mm + hft for mm, hft in zip(mm_counts, hft_counts)]
mm_share  = [mm / t if t > 0 else 0 for mm, t in zip(mm_counts, totals)]
hft_share = [1 - s for s in mm_share]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#111111')
for ax in axes:
    ax.set_facecolor('#111111')
    ax.tick_params(colors='white')
    ax.grid(True, linestyle='--', linewidth=0.4, alpha=0.5, color='#444444')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#444444')
    ax.spines['bottom'].set_color('#444444')

# Absolute fill counts
axes[0].bar(x - 0.2, mm_counts,  0.4, label='MM fills',  color='#4499ff', alpha=0.85)
axes[0].bar(x + 0.2, hft_counts, 0.4, label='HFT fills', color='#ff4444', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, color='white')
axes[0].set_title('Fill count by scenario', color='white', fontsize=12)
axes[0].set_ylabel('Number of fills', color='white')
axes[0].legend(facecolor='#222222', edgecolor='#444444', labelcolor='white')

# Stacked share
axes[1].bar(x, mm_share,  label='MM share',  color='#4499ff', alpha=0.85)
axes[1].bar(x, hft_share, bottom=mm_share, label='HFT share', color='#ff4444', alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(names, color='white')
axes[1].set_title('Fill share: MM vs HFT', color='white', fontsize=12)
axes[1].set_ylabel('Fraction of total fills', color='white')
axes[1].legend(facecolor='#222222', edgecolor='#444444', labelcolor='white')

plt.suptitle('Phase 3 — Fill share comparison across scenarios (3 days each)',
             color='white', fontsize=13)
plt.tight_layout()
plt.show()

## HFT state timeline — normal scenario

In [ ]:
log_normal = results['normal'][0]
stride = max(1, len(log_normal) // 3000)
s = log_normal.iloc[::stride]
t_hours = s['t'] / 3600

state_colors = {0: '#00ff88', 1: '#ffcc00', 2: '#ff9500', 3: '#ff4444'}
state_labels = {0: 'ACTIVE', 1: 'ONE_SIDED_BID', 2: 'ONE_SIDED_ASK', 3: 'OFFLINE'}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
fig.patch.set_facecolor('#111111')
for ax in (ax1, ax2):
    ax.set_facecolor('#111111')
    ax.tick_params(colors='white')
    ax.grid(True, linestyle='--', linewidth=0.4, alpha=0.4, color='#444444')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#444444')
    ax.spines['bottom'].set_color('#444444')

ax1.plot(t_hours, s['fair_mid'], color='#ffcc00', linewidth=0.8, label='Fair mid')
ax1.set_ylabel('EUR/USD', color='#ffcc00')
ax1.tick_params(axis='y', colors='#ffcc00')
ax1.set_title('Normal scenario — EUR/USD price with HFT state overlay', color='white', fontsize=12)

ylim = ax1.get_ylim()
for sv, sc in state_colors.items():
    mask = s['hft_state'].values == sv
    if mask.any():
        ax1.fill_between(t_hours.values, ylim[0], ylim[1],
                         where=mask, alpha=0.08, color=sc, label=state_labels[sv])
ax1.legend(facecolor='#222222', edgecolor='#444444', labelcolor='white', fontsize=8)

ax2.plot(t_hours, s['hft_state'], color='#4499ff', linewidth=0.7, drawstyle='steps-post')
ax2.set_yticks([0, 1, 2, 3])
ax2.set_yticklabels(['ACTIVE', 'ONE_SIDED_BID', 'ONE_SIDED_ASK', 'OFFLINE'],
                     color='white', fontsize=8)
ax2.set_ylabel('HFT state', color='white')
ax2.set_xlabel('Time (hours)', color='white')
ax2.set_title('HFT state timeline', color='white', fontsize=12)

plt.tight_layout()
plt.show()